# Table of Contents
- 1. Missing Data Taxonomy (MCAR, MAR, MNAR)
- 2. Advanced Imputation Strategies (KNN, Iterative, Rolling)
- 3. Outlier Handling via Winsorization & Clipping
- 4. Duplicate Detection & Entity Resolution Basics

In [ ]:
# Global Imports and Runtime Visual Configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from scipy.stats import mstats

# Visual styling settings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

print(
    "Global libraries for missing data analysis & imputation loaded"
    " successfully."
)

---
## 1. Missing Data Taxonomy (MCAR, MAR, MNAR)

### 1. Definition
Missing data taxonomy classifies missingness mechanisms into three categories based on their relationship with observed and unobserved variables: Missing Completely at Random (MCAR), Missing at Random (MAR), and Missing Not at Random (MNAR).

### 2. Math Formula
Let $Y$ represent complete data, partitioned into observed $Y_{obs}$ and missing $Y_{mis}$, and let $R$ be the missingness indicator matrix.
- **MCAR**: $P(R \vert{} Y_{obs}, Y_{mis}) = P(R)$
- **MAR**: $P(R \vert{} Y_{obs}, Y_{mis}) = P(R \vert{} Y_{obs})$
- **MNAR**: $P(R \vert{} Y_{obs}, Y_{mis})$ depends directly on $Y_{mis}$

### 3. Notation Pronunciation
- $Y_{mis}$: "Y missing"
- $R$: "Missingness indicator matrix"
- $P(R \vert{} Y_{obs})$: "Probability of missingness given observed data"

### 4. Naive Example
A survey where respondents accidentally skip a question due to a page glitch (MCAR), vs. younger respondents skipping income questions more often than older respondents (MAR), vs. high earners concealing their exact income (MNAR).

### 5. Uses
- Determining appropriate statistical techniques and imputation validity.
- Avoiding biased parameter estimates caused by improper deletion or imputation.
- Structuring data cleaning pipelines according to underlying missingness mechanisms.

In [ ]:
np.random.seed(42)
df_tax = pd.DataFrame({
    'age': np.random.randint(20, 60, 100),
    'income': np.random.normal(50000, 15000, 100)
})

# Introduce MAR missingness (income missing more often for younger ages)
df_tax.loc[df_tax['age'] < 30, 'income'] = np.nan

# --- KNN IMPUTATION ---
# Initialize the imputer (n_neighbors=5 means look at the 5 closest neighbors)
imputer = KNNImputer(n_neighbors=3)
# Fit and transform the dataframe
df_imputed = pd.DataFrame(imputer.fit_transform(df_tax), columns=df_tax.columns)

print(df_imputed[df_imputed['age'] < 30])

In [14]:

df_imp = pd.DataFrame({
    'feature_1': [1.0, 2.0, np.nan, 4.0, 5.0],
    'feature_2': [2.0, 4.0, 3.0, np.nan, 10.0]
})
print(df_imp)
print("_" * 60)
imputer = KNNImputer(n_neighbors=2)
df_imputed = pd.DataFrame(imputer.fit_transform(df_imp), columns=df_imp.columns)
print("Imputed Data:\n", df_imputed)

   feature_1  feature_2
0        1.0        2.0
1        2.0        4.0
2        NaN        3.0
3        4.0        NaN
4        5.0       10.0
____________________________________________________________
Imputed Data:
    feature_1  feature_2
0        1.0        2.0
1        2.0        4.0
2        1.5        3.0
3        4.0        7.0
4        5.0       10.0


---
## 3. Outlier Handling via Winsorization & Clipping

### 1. Definition
Outlier handling via winsorization and clipping caps extreme values at specific percentile thresholds (e.g., 1st and 99th percentiles) or hard boundaries instead of dropping rows, preserving sample size while reducing variance distortion.

### 2. Math Formula
Given sequence $X$, clipped value $X_{clip}$ is bounded between lower percentile $L_p$ and upper percentile $U_p$:
$$X_{clip} = \max(L_p, \min(X, U_p))$$

### 3. Notation Pronunciation
- $L_p$: "Lower percentile threshold"
- $U_p$: "Upper percentile threshold"
- `mstats.winsorize`: "Masked statistics winsorize function"

### 4. Naive Example
Capping extreme employee salaries exceeding the 99th percentile ceiling ($\$500,000$) down to that threshold so a few executive outliers do not distort linear regression coefficients.

### 5. Uses
- Stabilizing parametric models sensitive to extreme residuals.
- Preventing gradient explosion or distortion in machine learning pipelines.
- Retaining sample records without discarding extreme observations.

In [19]:
import numpy as np
from scipy.stats import mstats

data_outliers = [10, 12, 14, 15, 13, 11, 12, 100, -50] # 100 and -50 are outliers
# Convert list to numpy array so it has a .shape attribute
arr_outliers = np.array(data_outliers)
winsorized_data = mstats.winsorize(arr_outliers, limits=[0.12, 0.12]) # 1.2 is little higher than 1/9 = 0.111

print("Original:", data_outliers)
print("Winsorized:", winsorized_data.tolist())

Original: [10, 12, 14, 15, 13, 11, 12, 100, -50]
Winsorized: [10, 12, 14, 15, 13, 11, 12, 15, 10]


In [21]:
df_dup = pd.DataFrame({
    'customer_id': [101, 102, 101, 103],
    'name': ['Alice', 'Bob', 'Alice', 'Charlie'],
    'purchase_amt': [250, 150, 250, 300]
})

print(df_dup)

# Exact duplicate detection
duplicates_mask = df_dup.duplicated()
print("Duplicate rows mask:\n", duplicates_mask)
df_cleaned = df_dup.drop_duplicates()
print("\nDeduplicated DataFrame:\n", df_cleaned)

   customer_id     name  purchase_amt
0          101    Alice           250
1          102      Bob           150
2          101    Alice           250
3          103  Charlie           300
Duplicate rows mask:
 0    False
1    False
2     True
3    False
dtype: bool

Deduplicated DataFrame:
    customer_id     name  purchase_amt
0          101    Alice           250
1          102      Bob           150
3          103  Charlie           300


---
## Summary Table

| Missingness / Data Issue | Underlying Mechanism | Recommended Treatment Strategy | Primary Objective |
| :--- | :--- | :--- | :--- |
| **MCAR** | Missing completely at random | Listwise deletion or any imputation | Unbiased estimates without structural risk |
| **MAR & MNAR** | Missing dependent on observed/unobserved data | KNN Imputation, MICE, or explicit indicator flags | Preserving multivariate distribution |
| **Extreme Outliers** | Skewed measurement or data corruption | Winsorization & percentile clipping | Robust parameter estimation without sample loss |
| **Entity Duplicates** | Typos, inconsistent formatting, system merges | Exact deduplication & string similarity matching | Ensuring primary key uniqueness & data integrity |